# Titanic Dataset EDA and Preprocessing
**Author**: Randi Sumitro
**Project**: Membangun Sistem Machine Learning - Dicoding

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

In [ ]:
# Load Dataset
url = "https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Dataset Information
print("=== Dataset Information ===")
df.info()

print("\n=== Statistical Summary ===")
df.describe(include='all')

In [ ]:
# Check Missing Values
print("=== Missing Values ===")
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})
print(missing_df)

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
# Exploratory Data Analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Titanic Dataset EDA - Randi Sumitro', fontsize=16)

# 1. Survival Distribution
sns.countplot(data=df, x='Survived', ax=axes[0,0])
axes[0,0].set_title('Survival Distribution')
axes[0,0].set_xlabel('Survived (0=No, 1=Yes)')

# 2. Age Distribution
sns.histplot(data=df, x='Age', bins=30, kde=True, ax=axes[0,1])
axes[0,1].set_title('Age Distribution')

# 3. Fare Distribution
sns.histplot(data=df, x='Fare', bins=30, kde=True, ax=axes[0,2])
axes[0,2].set_title('Fare Distribution')

# 4. Survival by Sex
sns.countplot(data=df, x='Sex', hue='Survived', ax=axes[1,0])
axes[1,0].set_title('Survival by Sex')

# 5. Survival by Pclass
sns.countplot(data=df, x='Pclass', hue='Survived', ax=axes[1,1])
axes[1,1].set_title('Survival by Passenger Class')

# 6. Age vs Survival
sns.boxplot(data=df, x='Survived', y='Age', ax=axes[1,2])
axes[1,2].set_title('Age vs Survival')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Analysis
plt.figure(figsize=(12, 8))

# Select numeric columns for correlation
numeric_cols = ['Survived', 'Pclass', 'Age', 'Siblings/Spouses Aboard', 'Parents/Children Aboard', 'Fare']
correlation_matrix = df[numeric_cols].corr()

# Create heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix - Titanic Dataset', fontsize=14)
plt.tight_layout()
plt.show()

print("Correlation with Survival:")
print(correlation_matrix['Survived'].sort_values(ascending=False))

In [ ]:
# Preprocessing Class
class TitanicPreprocessor:
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        self.age_imputer = SimpleImputer(strategy='median')
        self.fare_imputer = SimpleImputer(strategy='median')
        self.feature_columns = ['Pclass', 'Sex', 'Age', 'Siblings/Spouses Aboard', 'Parents/Children Aboard', 'Fare']
        self.target_column = 'Survived'
        
    def fit(self, X, y=None):
        """Fit the preprocessor"""
        # Make a copy to avoid modifying original data
        X_processed = X.copy()
        
        # Encode Sex column
        X_processed['Sex'] = self.label_encoder.fit_transform(X_processed['Sex'])
        
        # Fit imputers
        self.age_imputer.fit(X_processed[['Age']])
        self.fare_imputer.fit(X_processed[['Fare']])
        
        # Impute missing values
        X_processed['Age'] = self.age_imputer.transform(X_processed[['Age']])
        X_processed['Fare'] = self.fare_imputer.transform(X_processed[['Fare']])
        
        # Fit scaler
        self.scaler.fit(X_processed[self.feature_columns])
        
        return self
    
    def transform(self, X):
        """Transform the data"""
        # Make a copy
        X_processed = X.copy()
        
        # Encode Sex column
        X_processed['Sex'] = self.label_encoder.transform(X_processed['Sex'])
        
        # Impute missing values
        X_processed['Age'] = self.age_imputer.transform(X_processed[['Age']])
        X_processed['Fare'] = self.fare_imputer.transform(X_processed[['Fare']])
        
        # Scale features
        X_processed[self.feature_columns] = self.scaler.transform(X_processed[self.feature_columns])
        
        return X_processed
    
    def fit_transform(self, X, y=None):
        """Fit and transform the data"""
        return self.fit(X, y).transform(X)

print("Preprocessing class defined successfully!")

In [ ]:
# Apply Preprocessing
# Initialize preprocessor
preprocessor = TitanicPreprocessor()

# Split features and target
X = df[preprocessor.feature_columns]
y = df[preprocessor.target_column]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {preprocessor.feature_columns}")

# Fit and transform the data
X_processed = preprocessor.fit_transform(X)

print("\nData after preprocessing:")
print(f"Processed features shape: {X_processed.shape}")
print(f"Sample processed data:\n{X_processed.head()}")

In [ ]:
# Save Processed Data
import os
import pickle

# Create directories
os.makedirs('data_preprocessed', exist_ok=True)

# Combine features and target
processed_df = X_processed.copy()
processed_df['Survived'] = y

# Save processed dataset
processed_df.to_csv('data_preprocessed/titanic_processed.csv', index=False)

# Save preprocessor object
with open('data_preprocessed/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

# Save data info
with open('data_preprocessed/data_info.txt', 'w') as f:
    f.write(f"Titanic Dataset Preprocessing Results\n")
    f.write(f"Author: Randi Sumitro\n")
    f.write(f"Date: {pd.Timestamp.now()}\n\n")
    f.write(f"Original shape: {df.shape}\n")
    f.write(f"Processed shape: {processed_df.shape}\n")
    f.write(f"Features: {preprocessor.feature_columns}\n")
    f.write(f"Target: {preprocessor.target_column}\n")
    f.write(f"Missing values handled: Age, Fare\n")
    f.write(f"Categorical encoded: Sex\n")
    f.write(f"Features scaled: StandardScaler\n")

print("Preprocessing completed successfully!")
print("Files saved:")
print("- data_preprocessed/titanic_processed.csv")
print("- data_preprocessed/preprocessor.pkl")
print("- data_preprocessed/data_info.txt")

# Verify saved files
print("\nVerification:")
print(f"Processed CSV shape: {pd.read_csv('data_preprocessed/titanic_processed.csv').shape}")
print(f"Files in directory: {os.listdir('data_preprocessed')}")

In [ ]:
# Final Summary
print("=== TITANIC DATASET PREPROCESSING COMPLETE ===")
print(f"Author: Randi Sumitro")
print(f"Processing Date: {pd.Timestamp.now()}")
print(f"\nDataset Summary:")
print(f"- Total Passengers: {len(df)}")
print(f"- Survived: {df['Survived'].sum()} ({df['Survived'].mean()*100:.1f}%)")
print(f"- Male: {(df['Sex']=='male').sum()} ({(df['Sex']=='male').mean()*100:.1f}%)")
print(f"- Female: {(df['Sex']=='female').sum()} ({(df['Sex']=='female').mean()*100:.1f}%)")
print(f"\nPreprocessing Steps:")
print(f"- Handled missing values in Age and Fare")
print(f"- Encoded categorical variable (Sex)")
print(f"- Scaled all features using StandardScaler")
print(f"- Saved processed data and preprocessor object")
print(f"\nFiles Generated:")
for file in os.listdir('data_preprocessed'):
    print(f"- {file}")